# Preproccessing - Pytorch

The following notebook applies preprocessing techniques using PyTorch. It uses the `California Housing` dataset.

First, import all the libraries that are necessary for this notebook.

In [64]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## Dataset

Let's load the data directly from sklearn as a DataFrame and inspect the first few rows to understand its features.

In [65]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


Let's use the `.info()` method to get a quick summary of our data 

In [66]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


Using `sklearn`, we split the data into training, validation, and test sets (80/10/10).

In [67]:
x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

Let's verify the split by checking the actual size of each dataset.

In [68]:
print(f"Train set: {len(x_train)}, {len(y_train)}")
print(f"Validation set: {len(x_val)}, {len(y_val)}")
print(f"Test set: {len(x_test)}, {len(y_test)}")

Train set: 16512, 16512
Validation set: 2064, 2064
Test set: 2064, 2064


Now, let's convert our data from pandas DataFrames to PyTorch tensors, which is required for model training.

In [69]:
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
x_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

## Preproccessing

To improve data quality, we will implement an outlier removal process using PyTorch. This approach follows the same methodology we covered in class, but adapted for tensor operations:

* Calculate the fisrt and third quartiles, and the interquartil range (IQR)
* Define the lower and upper bound for outliers
* Create a copy of the input tensor to avoid modifying the original data
* Cap values (excluiding `Latitude` and `Longitude` features) within the bounds.

In [70]:
class Outliers(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        features = x[:, :6]

        Q1 = torch.quantile(features, 0.25, dim=0)
        Q3 = torch.quantile(features, 0.75, dim=0)
        IQR = Q3 - Q1

        self.register_buffer('lower_bound', Q1 - 1.5 * IQR)
        self.register_buffer('upper_bound', Q3 + 1.5 * IQR)

    def forward(self, x) -> torch.Tensor:
        x_capped = x.clone()
        x_capped[:, :6].clip_(self.lower_bound, self.upper_bound) # Generated by AI to cap outliers 

        return x_capped


Now we apply standard scaling:

* Exclude `Latitude` and `Longitude` from scaling.
* Calculate the mean and standard devaition for each feature.
* Normalize each feature by subtracting its mean and dividing by its standard deviation

In [ ]:
class ScalingLayer(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        features = x[:, :6]

        self.register_buffer('mean', features.mean(dim=0))
        self.register_buffer('std', features.std(dim=0))

    def forward(self, x) -> torch.Tensor:
        return torch.cat([(x[:, :6] - self.mean) / self.std, x[:, 6:]], dim=1) # Generated by AI to apply standard scaling in a single step avoiding Latitude and Longitude

Now we combine both preprocessing steps into a single `PreprocessingLayer` class. This layer will sequentially apply outlier removal and standard scaling to prepare the data for model training.

In [72]:
class PreproccessingLayer(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.outliers = Outliers(x)
        x_capped = self.outliers(x)

        self.scaler = ScalingLayer(x_capped)

    def forward(self, x) -> torch.Tensor:
        return self.scaler(self.outliers(x))

## Model Training and Evaluation

We implement the training loop function. This function handles model training with early stopping, following the same logic from the Feed Forward Network activity.

In [73]:
def training_loop(model, optimizer, loss_fn, dataloader, epochs=100, delta = 0.005, patience=3):
    model.train()

    epoch_val_loss = []

    for epoch in range(epochs):
        epoch_loss = 0.0

        for X_batch, y_batch in dataloader:
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)

            epoch_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        avg_loss = epoch_loss / len(dataloader)
            
        # Early stopping 
        epoch_val_loss.append(avg_loss)
        if (epoch >= patience):
            if abs(epoch_val_loss[epoch - patience] - epoch_val_loss[epoch]) < delta:  
                if epoch_val_loss[epoch - patience] >= epoch_val_loss[epoch]:
                    print("The model is not improving, early stopping triggered.")
                    break

        print (f'Epoch {epoch+1}, Loss: {avg_loss:.4f}, Loss (MSE): {avg_loss:.4f}')

We define the MSE (Mean Squared Error) loss function, suitable for regression tasks.

In [74]:
loss_fn = nn.MSELoss()

We create a `DataLoader` for the training set, which organizes the data into batches and enables shuffling for better training convergence.

In [75]:
dataset = TensorDataset(x_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

### First Model

The first model is configured with the following hyperparameters:

Arquitecture:
* Input Layer = 8
* Hidden layer = 64
* Dropout = 0.2
* Hidden layer = 28 
* Output layer = 1

Training Configurations:
* Epochs = 100
* Bacth size = 32
* Optimizer = Adam
* Learning rate = 0.001
* Delta = 0.005
* Patience = 3

In [78]:
class Model_1(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.preproccessing = PreproccessingLayer(x)
        self.network = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 28),
            nn.ReLU(),
            nn.Linear(28, 1)
        )

    def forward(self, x) -> torch.Tensor:
        x = self.preproccessing(x)
        return self.network(x)

In [79]:
model_1 = Model_1(x_train_tensor)
optimizer = optim.Adam(model_1.parameters(), lr=0.001)
training_loop(model_1, optimizer, loss_fn, dataloader)

Epoch 1, Loss: 1.3071, Loss (MSE): 1.3071
Epoch 2, Loss: 0.7467, Loss (MSE): 0.7467
Epoch 3, Loss: 0.5998, Loss (MSE): 0.5998
Epoch 4, Loss: 0.5423, Loss (MSE): 0.5423
Epoch 5, Loss: 0.5210, Loss (MSE): 0.5210
Epoch 6, Loss: 0.5063, Loss (MSE): 0.5063
Epoch 7, Loss: 0.4956, Loss (MSE): 0.4956
Epoch 8, Loss: 0.4878, Loss (MSE): 0.4878
Epoch 9, Loss: 0.4744, Loss (MSE): 0.4744
Epoch 10, Loss: 0.4739, Loss (MSE): 0.4739
Epoch 11, Loss: 0.4628, Loss (MSE): 0.4628
Epoch 12, Loss: 0.4632, Loss (MSE): 0.4632
Epoch 13, Loss: 0.4536, Loss (MSE): 0.4536
Epoch 14, Loss: 0.4529, Loss (MSE): 0.4529
Epoch 15, Loss: 0.4495, Loss (MSE): 0.4495
Epoch 16, Loss: 0.4453, Loss (MSE): 0.4453
Epoch 17, Loss: 0.4453, Loss (MSE): 0.4453
Epoch 18, Loss: 0.4414, Loss (MSE): 0.4414
Epoch 19, Loss: 0.4392, Loss (MSE): 0.4392
Epoch 20, Loss: 0.4387, Loss (MSE): 0.4387
Epoch 21, Loss: 0.4351, Loss (MSE): 0.4351
Epoch 22, Loss: 0.4324, Loss (MSE): 0.4324
Epoch 23, Loss: 0.4336, Loss (MSE): 0.4336
The model is not imp

### Second Model

The second model is configured with the following hyperparameters:

Arquitecture:
* Input Layer = 8
* Hidden layer = 128
* Dropout = 0.3
* Hidden layer = 64
* Dropout = 0.2
* Hidden layer = 28 
* Output layer = 1

Training Configurations:
* Epochs = 100
* Bacth size = 32
* Optimizer = Adam
* Learning rate = 0.001
* Delta = 0.001
* Patience = 5

In [81]:
class Model_2(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.preproccessing = PreproccessingLayer(x)
        self.network = nn.Sequential(
            nn.Linear(8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 28),
            nn.ReLU(),
            nn.Linear(28, 1)
        )

    def forward(self, x) -> torch.Tensor:
        x = self.preproccessing(x)
        return self.network(x)

In [93]:
model_2 = Model_2(x_train_tensor)
optimizer = optim.Adam(model_2.parameters(), lr=0.001)
training_loop(model_2, optimizer, loss_fn, dataloader, delta=0.001, patience=5)

Epoch 1, Loss: 1.4700, Loss (MSE): 1.4700
Epoch 2, Loss: 0.7326, Loss (MSE): 0.7326
Epoch 3, Loss: 0.6100, Loss (MSE): 0.6100
Epoch 4, Loss: 0.5777, Loss (MSE): 0.5777
Epoch 5, Loss: 0.5409, Loss (MSE): 0.5409
Epoch 6, Loss: 0.5259, Loss (MSE): 0.5259
Epoch 7, Loss: 0.5117, Loss (MSE): 0.5117
Epoch 8, Loss: 0.5052, Loss (MSE): 0.5052
Epoch 9, Loss: 0.4950, Loss (MSE): 0.4950
Epoch 10, Loss: 0.4796, Loss (MSE): 0.4796
Epoch 11, Loss: 0.4789, Loss (MSE): 0.4789
Epoch 12, Loss: 0.4692, Loss (MSE): 0.4692
Epoch 13, Loss: 0.4650, Loss (MSE): 0.4650
Epoch 14, Loss: 0.4629, Loss (MSE): 0.4629
Epoch 15, Loss: 0.4607, Loss (MSE): 0.4607
Epoch 16, Loss: 0.4583, Loss (MSE): 0.4583
Epoch 17, Loss: 0.4562, Loss (MSE): 0.4562
Epoch 18, Loss: 0.4546, Loss (MSE): 0.4546
Epoch 19, Loss: 0.4471, Loss (MSE): 0.4471
Epoch 20, Loss: 0.4525, Loss (MSE): 0.4525
Epoch 21, Loss: 0.4542, Loss (MSE): 0.4542
Epoch 22, Loss: 0.4446, Loss (MSE): 0.4446
Epoch 23, Loss: 0.4476, Loss (MSE): 0.4476
Epoch 24, Loss: 0.44

### Third Model

The third model is configured with the following hyperparameters:

Arquitecture:
* Input Layer = 8
* Hidden layer = 256
* Dropout = 0.4
* Hidden layer = 128
* Dropout = 0.3
* Hidden layer = 64
* Dropout = 0.2
* Hidden layer = 28 
* Output layer = 1

Training Configurations:
* Epochs = 100
* Bacth size = 32
* Optimizer = Adam
* Learning rate = 0.001
* Delta = 0.001
* Patience = 10

In [94]:
class Model_3(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.preproccessing = PreproccessingLayer(x)
        self.network = nn.Sequential(
            nn.Linear(8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 28),
            nn.ReLU(),
            nn.Linear(28, 1)
        )

    def forward(self, x) -> torch.Tensor:
        x = self.preproccessing(x)
        return self.network(x)

In [95]:
model_3 = Model_3(x_train_tensor)
optimizer = optim.Adam(model_3.parameters(), lr=0.001)
training_loop(model_3, optimizer, loss_fn, dataloader, delta=0.001, patience=10)

Epoch 1, Loss: 1.4329, Loss (MSE): 1.4329
Epoch 2, Loss: 0.8026, Loss (MSE): 0.8026
Epoch 3, Loss: 0.6483, Loss (MSE): 0.6483
Epoch 4, Loss: 0.6000, Loss (MSE): 0.6000
Epoch 5, Loss: 0.5732, Loss (MSE): 0.5732
Epoch 6, Loss: 0.5398, Loss (MSE): 0.5398
Epoch 7, Loss: 0.5299, Loss (MSE): 0.5299
Epoch 8, Loss: 0.5087, Loss (MSE): 0.5087
Epoch 9, Loss: 0.5057, Loss (MSE): 0.5057
Epoch 10, Loss: 0.4920, Loss (MSE): 0.4920
Epoch 11, Loss: 0.4902, Loss (MSE): 0.4902
Epoch 12, Loss: 0.4841, Loss (MSE): 0.4841
Epoch 13, Loss: 0.4761, Loss (MSE): 0.4761
Epoch 14, Loss: 0.4726, Loss (MSE): 0.4726
Epoch 15, Loss: 0.4674, Loss (MSE): 0.4674
Epoch 16, Loss: 0.4633, Loss (MSE): 0.4633
Epoch 17, Loss: 0.4637, Loss (MSE): 0.4637
Epoch 18, Loss: 0.4586, Loss (MSE): 0.4586
Epoch 19, Loss: 0.4606, Loss (MSE): 0.4606
Epoch 20, Loss: 0.4553, Loss (MSE): 0.4553
Epoch 21, Loss: 0.4518, Loss (MSE): 0.4518
Epoch 22, Loss: 0.4507, Loss (MSE): 0.4507
Epoch 23, Loss: 0.4552, Loss (MSE): 0.4552
Epoch 24, Loss: 0.44

Next, we define the `evaluate()` function to compute and compare the Mean Squared Error (MSE) of all three models on the validation set.

In [109]:
def evaluate(model, x, y, model_name="Model"):
    model.eval()
    with torch.no_grad():
        predictions = model(x)
        mse = loss_fn(predictions, y).item()
        print(f'{model_name} MSE: {mse:.4f}')

The following output displays the validation set MSE for all three models:

In [106]:
print("Validation set evaluation:")
evaluate(model_1, x_val_tensor, y_val_tensor, "Model 1")
evaluate(model_2, x_val_tensor, y_val_tensor, "Model 2")
evaluate(model_3, x_val_tensor, y_val_tensor, "Model 3")

Validation set evaluation:
Model 1 MSE: 0.5367
Model 2 MSE: 0.5913
Model 3 MSE: 0.4830


Based on the validation results, Model 3 achieved the lowest MSE. Let's now evaluate its performance on the test set to confirm generalization.

In [111]:
print("Test set evaluation:")
evaluate(model_3, x_test_tensor, y_test_tensor, "Model 3")

Test set evaluation:
Model 3 MSE: 0.4757
